## import relevant utility functions/objects:

In [2]:
import flow.networks as networks

print(networks.__all__)

from flow.networks import RingNetwork

# ring road network class
network_name = RingNetwork

# input parameter classes to the network class
from flow.core.params import NetParams, InitialConfig

# name of the network
name = "adversarial_training_example"

# network-specific parameters
from flow.networks.ring import ADDITIONAL_NET_PARAMS
net_params = NetParams(additional_params=ADDITIONAL_NET_PARAMS)

# initial configuration to vehicles
initial_config = InitialConfig(spacing="uniform", perturbation=1)

print('Flow utilities imported.')

['Network', 'BayBridgeNetwork', 'BayBridgeTollNetwork', 'BottleneckNetwork', 'FigureEightNetwork', 'TrafficLightGridNetwork', 'HighwayNetwork', 'RingNetwork', 'MergeNetwork', 'MultiRingNetwork', 'MiniCityNetwork', 'HighwayRampsNetwork', 'I210SubNetwork']
Flow utilities imported.


In [3]:
# vehicles class
from flow.core.params import VehicleParams

# vehicles dynamics models
from flow.controllers import IDMController, ContinuousRouter

# defines a wrapper to handle the RL inputs:
from flow.controllers import RLController


vehicles = VehicleParams()
vehicles.add("human",
             acceleration_controller=(IDMController, {}),
             routing_controller=(ContinuousRouter, {}),
             num_vehicles=21)

vehicles.add(veh_id="rl",
             acceleration_controller=(RLController, {}),
             routing_controller=(ContinuousRouter, {}),
             num_vehicles=1)

print('Vehicle controllers defined.')

Vehicle controllers defined.


## Import environment objects/functions:

In [4]:
from flow.core.params import SumoParams

sim_params = SumoParams(sim_step=0.1, render=False)

from flow.core.params import EnvParams

# Define horizon as a variable to ensure consistent use across notebook
HORIZON=100

env_params = EnvParams(
    # length of one rollout
    horizon=HORIZON,

    additional_params={
        # maximum acceleration of autonomous vehicles
        "max_accel": 1,
        # maximum deceleration of autonomous vehicles
        "max_decel": 5.0,
        # bounds on the ranges of ring road lengths the autonomous vehicle 
        # is trained on
        "ring_length": [220, 270],
    },
)

print('Environmental training parameters defined.')

Environmental training parameters defined.


In [10]:
import flow.envs as flowenvs

print(flowenvs.__all__)

from flow.envs.ring.wave_attenuation import WaveAttenuationPOEnv

# env_name = WaveAttenuationPOEnv

# print('environment defined.')

['Env', 'AccelEnv', 'LaneChangeAccelEnv', 'LaneChangeAccelPOEnv', 'TrafficLightGridTestEnv', 'MergePOEnv', 'BottleneckEnv', 'BottleneckAccelEnv', 'WaveAttenuationEnv', 'WaveAttenuationPOEnv', 'TrafficLightGridEnv', 'TrafficLightGridPOEnv', 'TrafficLightGridBenchmarkEnv', 'BottleneckDesiredVelocityEnv', 'TestEnv', 'BayBridgeEnv', 'SingleStraightRoad', 'BottleNeckAccelEnv', 'DesiredVelocityEnv', 'PO_TrafficLightGridEnv', 'GreenWaveTestEnv']


In [11]:
from flow.envs.ring.wave_attenuation import WaveAttenuationPOEnv,WaveAttenuationEnv

class WavePropagationPOEnv(WaveAttenuationEnv):
    """POMDP version of WaveAttenuationEnv.


    This environment is updated to focus on creating traffic waves, rather
    than dissipating them. This is done through updates to the compute_reward funciton.

    Note that this environment only works when there is one autonomous vehicle
    on the network.

    Required from env_params:

    * max_accel: maximum acceleration of autonomous vehicles
    * max_decel: maximum deceleration of autonomous vehicles
    * ring_length: bounds on the ranges of ring road lengths the autonomous
      vehicle is trained on

    States
        The state consists of the speed and headway of the ego vehicle, as well
        as the difference in speed between the ego vehicle and its leader.
        There is no assumption on the number of vehicles in the network.

    Actions
        See parent class

    Rewards
        See parent class

    Termination
        See parent class

    """

    @property
    def observation_space(self):
        """See class definition."""
        return Box(low=-float('inf'), high=float('inf'),
                   shape=(3, ), dtype=np.float32)

    
    
    def compute_reward(self, rl_actions, **kwargs):
        """See class definition."""
        # in the warmup steps
        if rl_actions is None:
            return 0

        vel = np.array([
            self.k.vehicle.get_speed(veh_id)
            for veh_id in self.k.vehicle.get_ids()
        ])
        
        if any(vel < -100) or kwargs['fail']:
            return 0.

        # reward velocity volitility:
        eta_2 = 4.
        reward = eta_2 * np.std(vel) / 20
        
        # punish getting too close to danger:
        rl_id = self.k.vehicle.get_rl_ids()[0]
        lead_id = self.k.vehicle.get_leader(rl_id) or rl_id
        
        rl_headway = self.k.vehicle.get_headway(rl_id)
    
        rl_speed_difference = (self.k.vehicle.get_speed(lead_id) - self.k.vehicle.get_speed(rl_id))
        
        rl_ttc = -rl_speed_difference/rl_headway
        
        eta = -10.0  # 0.25
        ttc_threshold = 2.0

        if rl_ttc < ttc_threshold:
            reward += eta * (rl_ttc)

        return float(reward)
    
    def get_state(self):
        """See class definition."""
        rl_id = self.k.vehicle.get_rl_ids()[0]
        lead_id = self.k.vehicle.get_leader(rl_id) or rl_id

        # normalizers
        max_speed = 30.0
        if self.env_params.additional_params['ring_length'] is not None:
            max_length = self.env_params.additional_params['ring_length'][1]
        else:
            max_length = self.k.network.length()

        observation = np.array([
            self.k.vehicle.get_speed(rl_id) / max_speed,
            (self.k.vehicle.get_speed(lead_id) -
             self.k.vehicle.get_speed(rl_id)) / max_speed,
            (self.k.vehicle.get_x_by_id(lead_id) -
             self.k.vehicle.get_x_by_id(rl_id)) % self.k.network.length()
            / max_length
        ])

        return observation

    def additional_command(self):
        """Define which vehicles are observed for visualization purposes."""
        # specify observed vehicles
        rl_id = self.k.vehicle.get_rl_ids()[0]
        lead_id = self.k.vehicle.get_leader(rl_id) or rl_id
        self.k.vehicle.set_observed(lead_id)
        
env_name = WavePropagationPOEnv
        
print('Adversarial environment initialized.')

Adversarial environment initialized.


In [12]:
# Creating flow_params. Make sure the dictionary keys are as specified. 
flow_params = dict(
    # name of the experiment
    exp_tag=name,
    # name of the flow environment the experiment is running on
    env_name=env_name,
    # name of the network class the experiment uses
    network=network_name,
    # simulator that is used by the experiment
    simulator='traci',
    # simulation-related parameters
    sim=sim_params,
    # environment related parameters (see flow.core.params.EnvParams)
    env=env_params,
    # network-related parameters (see flow.core.params.NetParams and
    # the network's documentation or ADDITIONAL_NET_PARAMS component)
    net=net_params,
    # vehicles to be placed in the network at the start of a rollout 
    # (see flow.core.vehicles.Vehicles)
    veh=vehicles,
    # (optional) parameters affecting the positioning of vehicles upon 
    # initialization/reset (see flow.core.params.InitialConfig)
    initial=initial_config
)

print('flow parameters defined.')

flow parameters defined.


In [14]:
import json

import ray
try:
    from ray.rllib.agents.agent import get_agent_class
except ImportError:
    from ray.rllib.agents.registry import get_agent_class
from ray.tune import run_experiments
from ray.tune.registry import register_env

from flow.utils.registry import make_create_env
from flow.utils.rllib import FlowParamsEncoder

print('Extra utility functions loaded.')

Extra utility functions loaded.


In [15]:
# number of parallel workers
N_CPUS = 2
# number of rollouts per training iteration
N_ROLLOUTS = 1

ray.init(num_cpus=N_CPUS)

RayContext(dashboard_url='', python_version='3.7.3', ray_version='1.12.1', ray_commit='4863e33856b54ccf8add5cbe75e41558850a1b75', address_info={'node_ip_address': '127.0.0.1', 'raylet_ip_address': '127.0.0.1', 'redis_address': None, 'object_store_address': '/tmp/ray/session_2023-11-27_14-23-43_777883_24493/sockets/plasma_store', 'raylet_socket_name': '/tmp/ray/session_2023-11-27_14-23-43_777883_24493/sockets/raylet', 'webui_url': '', 'session_dir': '/tmp/ray/session_2023-11-27_14-23-43_777883_24493', 'metrics_export_port': 56354, 'gcs_address': '127.0.0.1:63721', 'address': '127.0.0.1:63721', 'node_id': 'd67dc5347d1083630e9694d96c38c7364f8ab2680a048e841e3353ad'})

# Train adversarial agent:

In [18]:
# This is taken directly from the flow tutorial:


# The algorithm or model to train. This may refer to "
#      "the name of a built-on algorithm (e.g. RLLib's DQN "
#      "or PPO), or a user-defined trainable function or "
#      "class registered in the tune registry.")
alg_run = "PPO"

# agent_cls = get_agent_class(alg_run) # apparently depreciated

agent_cls = ray.rllib.agents.registry.get_trainer_class(alg_run)


config = agent_cls._default_config.copy()
config["num_workers"] = N_CPUS - 1  # number of parallel workers
config["train_batch_size"] = HORIZON * N_ROLLOUTS  # batch size
config["gamma"] = 0.999  # discount rate
config["model"].update({"fcnet_hiddens": [16, 16]})  # size of hidden layers in network
config["use_gae"] = True  # using generalized advantage estimation
config["lambda"] = 0.97  
config["sgd_minibatch_size"] = min(16 * 1024, config["train_batch_size"])  # stochastic gradient descent
config["kl_target"] = 0.02  # target KL divergence
config["num_sgd_iter"] = 10  # number of SGD iterations
config["horizon"] = HORIZON  # rollout horizon

# save the flow params for replay
flow_json = json.dumps(flow_params, cls=FlowParamsEncoder, sort_keys=True,
                       indent=4)  # generating a string version of flow_params
config['env_config']['flow_params'] = flow_json  # adding the flow_params to config dict
config['env_config']['run'] = alg_run

# Call the utility function make_create_env to be able to 
# register the Flow env for this experiment
create_env, gym_name = make_create_env(params=flow_params, version=0)

# Register as rllib env with Gym
register_env(gym_name, create_env)

In [19]:
trials = run_experiments({
    flow_params["exp_tag"]: {
        "run": alg_run,
        "env": gym_name,
        "config": {
            **config
        },
        "checkpoint_freq": 1,  # number of iterations between checkpoints
        "checkpoint_at_end": True,  # generate a checkpoint at the end
        "max_failures": 999,
        "stop": {  # stopping conditions
            "training_iteration": 1,  # number of iterations to stop after
        },
    },
})

print('Finished training.')

2023-11-27 14:26:11,188	INFO trial_runner.py:803 -- starting PPO_WavePropagationPOEnv-v0_33f28_00000
(PPOTrainer pid=24939) 2023-11-27 14:26:18,820	INFO trainer.py:2296 -- Your framework setting is 'tf', meaning you are using static-graph mode. Set framework='tf2' to enable eager execution with tf2.x. You may also then want to set eager_tracing=True in order to reach similar execution speed as with static-graph mode.
(PPOTrainer pid=24939) 2023-11-27 14:26:19,700	WARNING ppo.py:249 -- `train_batch_size` (100) cannot be achieved with your other settings (num_workers=1 num_envs_per_worker=1 rollout_fragment_length=200)! Auto-adjusting `rollout_fragment_length` to 100.
(PPOTrainer pid=24939) 2023-11-27 14:26:19,700	INFO ppo.py:269 -- In multi-agent mode, policies will be optimized sequentially by the multi-GPU optimizer. Consider setting simple_optimizer=True if this doesn't work for you.
(PPOTrainer pid=24939) 2023-11-27 14:26:19,700	INFO trainer.py:867 -- Current log_level is WARN. For 

Trial name,status,loc
PPO_WavePropagationPOEnv-v0_33f28_00000,RUNNING,


2023-11-27 14:26:26,406	ERROR trial_runner.py:876 -- Trial PPO_WavePropagationPOEnv-v0_33f28_00000: Error processing event.
NoneType: None
2023-11-27 14:26:26,408	INFO trial_runner.py:1241 -- Trial PPO_WavePropagationPOEnv-v0_33f28_00000: Attempting to restore trial state from last checkpoint.


Trial name,status,loc
PPO_WavePropagationPOEnv-v0_33f28_00000,PENDING,
Trial name,# failures,error file
PPO_WavePropagationPOEnv-v0_33f28_00000,1,/Users/vanderbilt/ray_results/adversarial_training_example/PPO_WavePropagationPOEnv-v0_33f28_00000_0_2023-11-27_14-26-11/error.txt


2023-11-27 14:26:26,418	ERROR ray_trial_executor.py:103 -- An exception occurred when trying to stop the Ray actor:Traceback (most recent call last):
  File "/opt/anaconda3/envs/anti_flow/lib/python3.7/site-packages/ray/tune/ray_trial_executor.py", line 93, in post_stop_cleanup
    ray.get(future, timeout=0)
  File "/opt/anaconda3/envs/anti_flow/lib/python3.7/site-packages/ray/_private/client_mode_hook.py", line 105, in wrapper
    return func(*args, **kwargs)
  File "/opt/anaconda3/envs/anti_flow/lib/python3.7/site-packages/ray/worker.py", line 1811, in get
    raise value
ray.exceptions.RayActorError: The actor died because of an error raised in its creation task, ray::PPOTrainer.__init__() (pid=24939, ip=127.0.0.1, repr=PPOTrainer)
  File "/opt/anaconda3/envs/anti_flow/lib/python3.7/site-packages/ray/rllib/agents/trainer.py", line 1035, in _init
    raise NotImplementedError
NotImplementedError

During handling of the above exception, another exception occurred:

ray::PPOTrainer.__i

...

(PPOTrainer pid=26632) 2023-11-27 14:32:10,245	ERROR worker.py:449 -- Exception raised in creation task: The actor died because of an error raised in its creation task, ray::PPOTrainer.__init__() (pid=26632, ip=127.0.0.1, repr=PPOTrainer)
(PPOTrainer pid=26632)   File "/opt/anaconda3/envs/anti_flow/lib/python3.7/site-packages/ray/rllib/agents/trainer.py", line 1035, in _init
(PPOTrainer pid=26632)     raise NotImplementedError
(PPOTrainer pid=26632) NotImplementedError
(PPOTrainer pid=26632) 
(PPOTrainer pid=26632) During handling of the above exception, another exception occurred:
(PPOTrainer pid=26632) 
(PPOTrainer pid=26632) ray::PPOTrainer.__init__() (pid=26632, ip=127.0.0.1, repr=PPOTrainer)
(PPOTrainer pid=26632)   File "/opt/anaconda3/envs/anti_flow/lib/python3.7/site-packages/ray/rllib/agents/trainer.py", line 831, in __init__
(PPOTrainer pid=26632)     config, logger_creator, remote_checkpoint_dir, sync_function_tpl
(PPOTrainer pid=26632)   File "/opt/anaconda3/envs/anti_flow/

Trial name,status,loc
PPO_WavePropagationPOEnv-v0_33f28_00000,RUNNING,
Trial name,# failures,error file
PPO_WavePropagationPOEnv-v0_33f28_00000,28,/Users/vanderbilt/ray_results/adversarial_training_example/PPO_WavePropagationPOEnv-v0_33f28_00000_0_2023-11-27_14-26-11/error.txt


Trial name,status,loc
PPO_WavePropagationPOEnv-v0_33f28_00000,RUNNING,
Trial name,# failures,error file
PPO_WavePropagationPOEnv-v0_33f28_00000,28,/Users/vanderbilt/ray_results/adversarial_training_example/PPO_WavePropagationPOEnv-v0_33f28_00000_0_2023-11-27_14-26-11/error.txt


(PPOTrainer pid=26694) 2023-11-27 14:32:23,242	ERROR worker.py:449 -- Exception raised in creation task: The actor died because of an error raised in its creation task, ray::PPOTrainer.__init__() (pid=26694, ip=127.0.0.1, repr=PPOTrainer)
(PPOTrainer pid=26694)   File "/opt/anaconda3/envs/anti_flow/lib/python3.7/site-packages/ray/rllib/agents/trainer.py", line 1035, in _init
(PPOTrainer pid=26694)     raise NotImplementedError
(PPOTrainer pid=26694) NotImplementedError
(PPOTrainer pid=26694) 
(PPOTrainer pid=26694) During handling of the above exception, another exception occurred:
(PPOTrainer pid=26694) 
(PPOTrainer pid=26694) ray::PPOTrainer.__init__() (pid=26694, ip=127.0.0.1, repr=PPOTrainer)
(PPOTrainer pid=26694)   File "/opt/anaconda3/envs/anti_flow/lib/python3.7/site-packages/ray/rllib/agents/trainer.py", line 831, in __init__
(PPOTrainer pid=26694)     config, logger_creator, remote_checkpoint_dir, sync_function_tpl
(PPOTrainer pid=26694)   File "/opt/anaconda3/envs/anti_flow/

Finished training.
